In [3]:
import recordlinkage
import pandas as pd
import re
import numpy as np

social_dataset = pd.read_excel('../../Mediated Schema Excels/socialmedia_schema.xlsx')

social_dataset = social_dataset[
    social_dataset['Other'].notnull() & (social_dataset['Other'].str.strip() != '')
]

def tokenize_and_clean_url(url):
    if type(url) is not str:
        url = str(url)
    url = url.lower().replace("http://", "").replace("https://", "").replace("disfold.com/company","").replace("www","").rstrip("/")
    tokens = re.split(r'\W+', url)
    tokens = sorted(token for token in tokens if token)
    return " ".join(tokens)

social_dataset['Other_clean'] = social_dataset['Other'].apply(tokenize_and_clean_url)

def social_blocking():
    indexer = recordlinkage.Index()
    indexer.sortedneighbourhood('Name', window=5)
    candidate_links = indexer.index(social_dataset)
    compare = recordlinkage.Compare()
    compare.string('Other_clean', 'Other_clean', method='jarowinkler', label='url_similarity')
    compare_vectors = compare.compute(candidate_links, social_dataset)
    
    matched_pairs = compare_vectors[compare_vectors['url_similarity'] > 0.83]

    n = len(matched_pairs)

    df = pd.DataFrame({
        "id": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(0)) + 2,
        "left_row_index": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(0)) + 2,
        "right_row_index": social_dataset.index.get_indexer(matched_pairs.index.get_level_values(1)) + 2,
        "left_name_company": social_dataset.loc[matched_pairs.index.get_level_values(0), "Name"].values,
        "right_name_company": social_dataset.loc[matched_pairs.index.get_level_values(1), "Name"].values,
        "left_industryname": [np.nan] * n,
        "right_industryname": [np.nan] * n,
        "left_sector": [np.nan] * n,
        "right_sector": [np.nan] * n,
        "left_address": [np.nan] * n,
        "right_address": [np.nan] * n,
        "left_city": [np.nan] * n,
        "right_city": [np.nan] * n,
        "left_state": [np.nan] * n,
        "right_state": [np.nan] * n,
        "left_country": [np.nan] * n,
        "right_country": [np.nan] * n,
        "left_continent": [np.nan] * n,
        "right_continent": [np.nan] * n,
        "left_other": social_dataset.loc[matched_pairs.index.get_level_values(0), "Other"].values,
        "right_other": social_dataset.loc[matched_pairs.index.get_level_values(1), "Other"].values,
    })


    df.to_csv("../../Deep Matcher/Blocking and Pairwise matching/socialmedia_blocking_for_deepmatcher.csv", index=False)

In [4]:
social_blocking()